### Imports

In [ ]:
%reload_ext autoreload
%autoreload 2


import sys, os
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path )
sys.path.append('your path')
sys.path = list(set(sys.path))
import numpy as np
import pandas as pd

import torch
import pytorch_lightning as pl
import wandb

import matplotlib.pyplot as plt

import scipy as sp
from scipy.ndimage.filters import gaussian_filter1d

from collections import Counter


import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
print(sys.path)
import myallsim.infer as infer
import myallsim.sim as sim
from myallsim.policies.policy import MELD, MELD_na, FIFO
from myallsim.outcome.counterfactual_inference import Inference_OrganITE
from myallsim.outcome.counterfactual_models import OrganITE_Network
from myallsim.data.myUNOSDataModule import myUNOSDataModule

import warnings
warnings.filterwarnings("ignore")

### Data loading

In [ ]:
data = 'UNOS'
data_dir = f'../data/{data}'
data_dir = "your path"

if data == 'UNOS':
    dm = myUNOSDataModule(data_dir,is_synth = True, batch_size=256)
    groups_r = [['ALCOHOL_HEAVY_DON_N', 'ALCOHOL_HEAVY_DON_U', 'ALCOHOL_HEAVY_DON_Y'],
                ['COD_CAD_DON_1.0', 'COD_CAD_DON_2.0', 'COD_CAD_DON_3.0', 'COD_CAD_DON_4.0', 'COD_CAD_DON_999.0'],
                ['DON_TY_C'], 
                ['ETHCAT_DON_1.0', 'ETHCAT_DON_2.0', 'ETHCAT_DON_4.0', 'ETHCAT_DON_5.0', 'ETHCAT_DON_6.0', 'ETHCAT_DON_7.0', 'ETHCAT_DON_9.0'], 
                ['GENDER_DON_F','GENDER_DON_M'],
                ['HIST_CANCER_DON_N', 'HIST_CANCER_DON_U', 'HIST_CANCER_DON_Y'],
                ['HIST_CIG_DON_N', 'HIST_CIG_DON_U', 'HIST_CIG_DON_Y'],
                ['NON_HRT_DON_N', 'NON_HRT_DON_Y']]
    groups_x = [
        ['DIAG_0.0', 'DIAG_1.0', 'DIAG_10.0', 'DIAG_11.0', 'DIAG_12.0', 'DIAG_13.0', 'DIAG_14.0', 'DIAG_15.0', 'DIAG_16.0',
       'DIAG_17.0', 'DIAG_18.0', 'DIAG_2.0', 'DIAG_3.0', 'DIAG_4.0',
       'DIAG_5.0', 'DIAG_6.0', 'DIAG_7.0', 'DIAG_8.0', 'DIAG_9.0'],
        ['ETHCAT_1.0', 'ETHCAT_2.0', 'ETHCAT_4.0', 'ETHCAT_5.0', 'ETHCAT_6.0', 'ETHCAT_7.0', 'ETHCAT_9.0'],
        ['GENDER_F', 'GENDER_M'],
        ['INIT_ASCITES_1.0', 'INIT_ASCITES_2.0', 'INIT_ASCITES_3.0', 'INIT_ASCITES_4.0'],
    ]

dm.prepare_data()
dm.setup(stage='fit')

DATA = pd.concat([dm._test_processed.copy(deep=True), dm._train_processed.copy(deep=True)])
DATA_true = DATA.copy(deep=True)
DATA_true[dm.real_cols] = dm.scaler.inverse_transform(
    DATA_true[dm.real_cols])

patients = DATA_true[dm.x_cols].copy(deep=True)
organs = DATA_true[dm.o_cols].copy(deep=True)

X = patients
O = organs
Y = dm._train_processed.Y


O_indices_in_real_cols = np.where(np.in1d(np.unique(dm.real_cols), O.columns.values))[0]
mean_real_cols_O = dm.scaler.mean_[O_indices_in_real_cols]
scale_real_cols_O = dm.scaler.scale_[O_indices_in_real_cols]
real_cols_indices_in_O = np.where(np.in1d(O.columns.values, np.unique(dm.real_cols)))[0]

X_indices_in_real_cols = np.where(np.in1d(np.unique(dm.real_cols), X.columns.values))[0]
mean_real_cols_X = dm.scaler.mean_[X_indices_in_real_cols]
scale_real_cols_X = dm.scaler.scale_[X_indices_in_real_cols]
real_cols_indices_in_X = np.where(np.in1d(X.columns.values, np.unique(dm.real_cols)))[0]

In [ ]:
O

In [ ]:
DATA

In [ ]:
# OrganITE
input_dim = len(dm.o_cols) + len(dm.x_cols)
hidden_dim=16
num_hidden_layers=4
output_dim=6
lr=0.007
gamma=0.79
lambd=0.15
kappa=0.15
weight_decay=0.0006
n_clusters=15
activation_type="leaky_relu"
dropout_prob=0
epochs = 100

organite_model = OrganITE_Network(
             input_dim=input_dim,
             hidden_dim=hidden_dim,
             num_hidden_layers=num_hidden_layers,
             output_dim=output_dim,
             lr=lr,
             gamma=gamma,
             lambd=lambd,
             kappa=kappa,
             weight_decay=weight_decay,
             n_clusters=n_clusters,
             activation_type=activation_type,
             dropout_prob=dropout_prob,
).double()

early_stopping = EarlyStopping('val_loss - total', patience = 2, verbose = True)

organite_trainer = pl.Trainer(callbacks=[early_stopping], max_epochs=epochs)
organite_trainer.fit(organite_model, datamodule=dm)

inference_oite = Inference_OrganITE(
    model=organite_model, mean=dm.mean, std=dm.std,
    x_indices_real=real_cols_indices_in_X, r_indices_real=real_cols_indices_in_O,
    x_mean=mean_real_cols_X, x_scale=scale_real_cols_X,
    r_mean=mean_real_cols_O, r_scale=scale_real_cols_O
)

In [ ]:
dm.setup(stage='test')
organite_trainer.test(organite_model, datamodule=dm)

In [ ]:
organ_cit_diff_test = np.array([O.iloc[0]])
organ_cit_diff_test[0][10] = 14
print(np.array([O.iloc[0]]))
print(organ_cit_diff_test)

print(inference_oite.infer(np.array([X.iloc[0]])))
print(inference_oite.infer(np.array([X.iloc[0]]), np.array([O.iloc[0]])))
print(inference_oite.infer(np.array([X.iloc[0]]), organ_cit_diff_test))

### KDEs

In [ ]:
O.shape

In [ ]:
kde_o = infer.KDEDensity()
kde_o.fit(O.loc[O["DON_TY_C"] == 1][:10000], one_hot_encoded=groups_r)

In [ ]:
kde_o.sample()

In [ ]:
X.shape

In [ ]:
kde_x = infer.KDEDensity()
kde_x.fit(X[:10000], one_hot_encoded=groups_x)

In [ ]:
kde_x.sample()

### Import acceptance model

In [ ]:
import numpy as np

import tensorflow as tf

# Instantiate the loss functions
binary_crossentropy = tf.keras.losses.BinaryCrossentropy()
mean_squared_error = tf.keras.losses.MeanSquaredError()

@tf.function  # This ensures TensorFlow works properly in graph mode
def custom_loss(y_true, y_pred):
    # y_true[:, 0] is the true values for ACC, y_pred[:, 0] is the predicted values for ACC
    acc_loss = binary_crossentropy(y_true[:, 0], y_pred[:, 0])
    
    # y_true[:, 1] is the true values for DEC_TIME, y_pred[:, 1] is the predicted values for DEC_TIME
    dec_time_loss = mean_squared_error(y_true[:, 1], y_pred[:, 1])
    
    # Combine the two losses (you can weight them differently if necessary)
    total_loss = acc_loss + dec_time_loss
    
    return total_loss

In [ ]:
# Load the model, passing the custom loss function to `custom_objects`
acceptance_model = tf.keras.models.load_model(
    'my_PatNet_models/my_PatNet_synth5.keras',
    custom_objects={'custom_loss': custom_loss},
    compile=False
)

acceptance_model.compile(optimizer='adam', loss=custom_loss)

In [ ]:
inference_oite = Inference_OrganITE(
    model=organite_model, mean=dm.mean, std=dm.std,
    x_indices_real=real_cols_indices_in_X, r_indices_real=real_cols_indices_in_O,
    x_mean=mean_real_cols_X, x_scale=scale_real_cols_X,
    r_mean=mean_real_cols_O, r_scale=scale_real_cols_O
)

In [ ]:
def transf(patient,organ):
        if (
            patient is None
            or real_cols_indices_in_X is None
            or scale_real_cols_X is None
            or mean_real_cols_X is None
        ):
            patient = patient
        else:
            patient[:, real_cols_indices_in_X] = (
                patient[:, real_cols_indices_in_X] - mean_real_cols_X
            ) / scale_real_cols_X

        if (
            organ is None
            or real_cols_indices_in_O is None
            or scale_real_cols_O is None
            or mean_real_cols_O is None
        ):
            organ = organ
        else:
            organ[:, real_cols_indices_in_O] = (
                organ[:, real_cols_indices_in_O] - mean_real_cols_O
            ) / scale_real_cols_O

        return patient, organ

In [ ]:
#load theta x,o to make outcome function 
theta_x = np.load('your path')
theta_o = np.load('your path')
v = 13.15
max = np.max(Y*dm.std + dm.mean)
def ite(pat,org):
    #print(pat, org)
    pat, org = transf(debug_x,debug_o)
    #print(pat, org)
    y0 = v*np.exp((pat @ theta_x + 0.5))
    y = v*np.exp((pat @ theta_x + (org @ theta_o) + 0.5))
    #print(y,y0)
    if y0<0:
        y0 = 0
    elif y0> max:
        y0 = max
    if y<0:
        y = 0
    elif y> max:
        y = max
    return int(y-y0)

In [ ]:
theta_o

In [ ]:
debug_x = np.array(kde_x.sample())
debug_o = kde_o.sample()
debug_o["COLD_ISCH"] = 0
debug_o = np.array(debug_o)
ite(debug_x,debug_o)

### Generate Simulated Acceptance dataset

In [ ]:
def synth_acceptance(x,o, hmeld = 0.01, hcit = 0.15, hage = 0.22, hbmi = 0.0, hrnd = 1, uncertainty = 1):
    bilirubin = x.INIT_BILIRUBIN
    inr = x.INIT_INR
    serum_creat = x.INIT_SERUM_CREAT
    
    MELD = 3.79 * (bilirubin if bilirubin > 0 else 0) + 11.2 * (inr if inr > 0 else 0) + 9.57 * (serum_creat if serum_creat > 0 else 0) + 6.43
    age_diff = abs(x["INIT_AGE"] - o["AGE_DON"])
    age_diff_cond = 1 if age_diff <= 30 else 0
    bmi_diff = abs(x["INIT_BMI_CALC"] - o["BMI_DON_CALC"])
    bmi_diff_cond = 1 if bmi_diff <= 5 else 0
    cit = o["COLD_ISCH"]
    if MELD > 40:
        MELD = 40
    #tot = MELD  - 2*cit - 5*np.random.rand() - age_diff_cond*20
    MELD_rel = MELD-20
    rnd = np.random.rand() - 0.5
    tot = hmeld*MELD_rel  - hcit*cit - hage*age_diff_cond*20 - hbmi*bmi_diff_cond + hrnd*rnd
    prob = 1/(1+np.exp(-tot/uncertainty))
    return prob

def synth_time(x,o, hh = 8, hsig = 1, hrnd = 0.4):
    acc = synth_acceptance(x,o)
    #time = (-8*acc**2 + 8*acc)
    time = -hh*acc*(acc-1)*np.exp(-(acc-0.5)**2/hsig) + hrnd*np.random.rand()
    return time

In [ ]:
## USED POLICY

def MELD_rank(Q):
    # DEFINITION OF (standard) MELD: https://en.wikipedia.org/wiki/Model_for_End-Stage_Liver_Disease#Determination
    bilirubin = Q["INIT_BILIRUBIN"].copy()
    inr = Q["INIT_INR"].copy()
    serum_creat = Q["INIT_SERUM_CREAT"].copy()

    bilirubin[bilirubin < 0] = 0
    inr[inr < 0] = 0
    serum_creat[serum_creat < 0] = 0
    
    Q["MELD"] = 3.79 * bilirubin + 11.2 * inr + 9.57 * serum_creat + 6.43

    Q = Q.sort_values(by='MELD', ascending=False)
    Q = Q.drop("MELD", axis = 1)
    return Q
        

In [ ]:
simulated_acceptance_dataset = pd.DataFrame(columns=list(X.columns) + list(O.columns) + ["ACC", "DEC_TIME"])
chains_lengths = []
org_cits = []

In [ ]:
generated_initial_organs = kde_o.sample(n=1000)
generated_initial_organs["COLD_ISCH"] = 0
generated_Qs = []
for organ in generated_initial_organs.iterrows():
    generated_Q = kde_x.sample(n=50)
    generated_Qs.append(generated_Q)

len(generated_Qs)

### MELD TEST

In [ ]:
meld_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    meld_ites = []
    meld_ranks = []
    meld_cits_alloc = []
    meld_cits = []
    meld_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = MELD_rank(generated_Q)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                meld_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                #meld_ites.append(ite(patient_np,organ_np))
                meld_ranks.append(rank)
                meld_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        meld_cits.append(organ["COLD_ISCH"])
        sim +=1
    meld_stats["mean_ites"].append(np.mean(meld_ites))
    meld_stats["mean_ranks"].append(np.mean(meld_ranks))
    meld_stats["mean_cits"].append(np.mean(meld_cits))
    meld_stats["mean_waste"].append((sim-len(meld_ranks))/sim)
        

In [ ]:
for stat in meld_stats:
    print(stat, np.mean(meld_stats[stat]), (np.std(meld_stats[stat])))


### MELD na 

In [ ]:
def MELDna_rank(Q):
    # DEFINITION OF (standard) MELD: https://en.wikipedia.org/wiki/Model_for_End-Stage_Liver_Disease#Determination
    bilirubin = Q["INIT_BILIRUBIN"].copy()
    inr = Q["INIT_INR"].copy()
    serum_creat = Q["INIT_SERUM_CREAT"].copy()
    serum_sodium = Q["INIT_SERUM_SODIUM"].copy()

    bilirubin[bilirubin < 0] = 0
    inr[inr < 0] = 0
    serum_creat[serum_creat < 0] = 0
    
    Q["MELD"] = 3.79 * bilirubin + 11.2 * inr + 9.57 * serum_creat + 6.43 + 1.59 * (135 - serum_sodium)

    Q = Q.sort_values(by='MELD', ascending=False)
    Q = Q.drop("MELD", axis = 1)
    return Q

In [ ]:
meldna_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    meldna_ites = []
    meldna_ranks = []
    meldna_cits_alloc = []
    meldna_cits = []
    meldna_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = MELDna_rank(generated_Q)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                meldna_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                
                meldna_ranks.append(rank)
                meldna_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        meldna_cits.append(organ["COLD_ISCH"])
        sim +=1
    meldna_stats["mean_ites"].append(np.mean(meldna_ites))
    meldna_stats["mean_ranks"].append(np.mean(meldna_ranks))
    meldna_stats["mean_cits"].append(np.mean(meldna_cits))
    meldna_stats["mean_waste"].append((sim-len(meldna_ranks))/sim)

In [ ]:
for stat in meldna_stats:
    print(stat, np.mean(meldna_stats[stat]), (np.std(meldna_stats[stat])))

### MaxAcc TEST

In [ ]:
for generated_Q in generated_Qs:
    generated_Q = generated_Q.drop(["MELD"], axis = 1)

In [ ]:
def MaxAcc_rank(Q, organ):
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q = Q.drop(["MELD"], axis = 1)
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    #print(Q_np)
    #print(organ_np)
    #print(Q_np.shape, organ_np.shape)
    Q_df = pd.DataFrame(Q_np)
    organ_df = pd.DataFrame(organ_np)
    conc = np.concatenate([Q_df, organ_df], axis=1)
    #print(conc.shape)
    

    Q["ACC"] = acceptance_model(conc)[:,0]

    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='ACC', ascending=False)
    Q = Q.drop("ACC", axis=1)
    return Q

In [ ]:
maxacc_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    maxacc_ites = []
    maxacc_ranks = []
    maxacc_cits_alloc = []
    maxacc_cits = []
    maxacc_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = MaxAcc_rank(generated_Q, organ)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                maxacc_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                
                maxacc_ranks.append(rank)
                maxacc_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        maxacc_cits.append(organ["COLD_ISCH"])
        sim +=1
    maxacc_stats["mean_ites"].append(np.mean(maxacc_ites))
    maxacc_stats["mean_ranks"].append(np.mean(maxacc_ranks))
    maxacc_stats["mean_cits"].append(np.mean(maxacc_cits))
    maxacc_stats["mean_waste"].append((sim-len(maxacc_ranks))/sim)
        

In [ ]:
for stat in maxacc_stats:
    print(stat, np.mean(maxacc_stats[stat]), (np.std(maxacc_stats[stat])))

### MinTime

In [ ]:
def MinTime_rank(Q, organ):
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q = Q.drop(["MELD"], axis = 1)
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    #print(Q_np)
    #print(organ_np)
    #print(Q_np.shape, organ_np.shape)
    Q_df = pd.DataFrame(Q_np)
    organ_df = pd.DataFrame(organ_np)
    conc = np.concatenate([Q_df, organ_df], axis=1)
    #print(conc.shape)
    

    Q["DEC_TIME"] = acceptance_model(conc)[:,1]

    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='DEC_TIME', ascending=True)
    Q = Q.drop("DEC_TIME", axis=1)
    return Q

In [ ]:
mintime_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    mintime_ites = []
    mintime_ranks = []
    mintime_cits_alloc = []
    mintime_cits = []
    mintime_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = MinTime_rank(generated_Q, organ)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                mintime_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                
                mintime_ranks.append(rank)
                mintime_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        mintime_cits.append(organ["COLD_ISCH"])
        sim +=1
    mintime_stats["mean_ites"].append(np.mean(mintime_ites))
    mintime_stats["mean_ranks"].append(np.mean(mintime_ranks))
    mintime_stats["mean_cits"].append(np.mean(mintime_cits))
    mintime_stats["mean_waste"].append((sim-len(mintime_ranks))/sim)

In [ ]:
for stat in mintime_stats:
    print(stat, np.mean(mintime_stats[stat]), (np.std(mintime_stats[stat])))

### Transplant Benefit

In [ ]:
for generated_Q in generated_Qs:
    generated_Q.drop(["MELD"], axis = 1)

In [ ]:
def TB_rank(Q, organ):
    Q = Q.drop(["MELD"], axis = 1)
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    # Calculate ITE
    Q["ITE"] = (inference_oite.infer(Q_np, organ_np) - inference_oite.infer(Q_np)).squeeze()

    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='ITE', ascending=False)
    Q = Q.drop("ITE", axis=1)
    return Q

In [ ]:
TB_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    TB_ites = []
    TB_ranks = []
    TB_cits_alloc = []
    TB_cits = []
    TB_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = TB_rank(generated_Q, organ)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                TB_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                
                TB_ranks.append(rank)
                TB_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        TB_cits.append(organ["COLD_ISCH"])
        sim +=1
    TB_stats["mean_ites"].append(np.mean(TB_ites))
    TB_stats["mean_ranks"].append(np.mean(TB_ranks))
    TB_stats["mean_cits"].append(np.mean(TB_cits))
    TB_stats["mean_waste"].append((sim-len(TB_ranks))/sim)
        

In [ ]:
for stat in TB_stats:
    print(stat, np.mean(TB_stats[stat]), (np.std(TB_stats[stat])))

### DYNA SCORES TEST

In [ ]:
def Dyna_rank(Q, organ, alpha1 = 0.5, alpha2 = 1, alpha3 = 0.6, iterations=5, topk = 10):
    Q = Q.drop(["MELD"], axis = 1)
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    Q_df = pd.DataFrame(Q_np)
    organ_df = pd.DataFrame(organ_np)
    conc = np.concatenate([Q_df, organ_df], axis=1)
    
    #print(conc.shape)
    accmdl = acceptance_model(conc)
    Q["ACC"] , Q["DEC_TIME"] = accmdl[:,0], accmdl[:,1]
    # Calculate ITE
    Q["ITE"] = (inference_oite.infer(Q_np, organ_np) - inference_oite.infer(Q_np)).squeeze()

    #Q["DYNA"] = Q["ACC"] * Q["ITE"] 
    Q["DYNA"] = (Q["ACC"]**alpha1) * (Q["ITE"]**alpha2) / (Q["DEC_TIME"]**alpha3)
    
    # print(Q)
    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='DYNA', ascending=False)

    Q = Q.drop("ITE", axis=1)
    Q = Q.drop("DEC_TIME", axis=1)
    Q = Q.drop("ACC", axis=1)
    Q = Q.drop("DYNA", axis=1)
    
    return Q

In [ ]:
DYNA_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
for run in range(10):
    print("run:",run)
    sim = 0
    DYNA_ites = []
    DYNA_ranks = []
    DYNA_cits_alloc = []
    DYNA_cits = []
    DYNA_waste = 0
    for organ in generated_initial_organs.iterrows():
        #print(sim)
        organ = organ[1]
        generated_Q = generated_Qs[sim]
        generated_Q = Dyna_rank(generated_Q, organ)
        rank = 0
        for patient in generated_Q.iterrows():
            rank += 1
            patient = patient[1]
            #combined_input = pd.concat([patient, organ], axis=0)
            #model_output  = acceptance_model(combined_input.to_frame().T)
            
            #accept = model_output[:,0].numpy()
            #print(accept)
            #time = model_output[:,1].numpy()
    
            accept = synth_acceptance(patient,organ)
            time = synth_time(patient,organ)
            #print("accept", accept)
            #print("time", time)
            
            ## update CIT
            organ["COLD_ISCH"] += time
            #XO = dict(organ) | dict(patient)
            
            if accept > np.random.rand():
                ## organ got accepted: add to obs, break sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                patient_np = np.array([patient])  # Convert patient Series to NumPy array
                organ_np = np.array([organ])       # Convert organ Series to NumPy array
                #print(patient)
                #print(organ)
                DYNA_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                
                DYNA_ranks.append(rank)
                DYNA_cits_alloc.append(organ["COLD_ISCH"])
                break
            #else:
                ## organ got rejected: add to obs, continue sim
                #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
        DYNA_cits.append(organ["COLD_ISCH"])
        sim +=1
    DYNA_stats["mean_ites"].append(np.mean(DYNA_ites))
    DYNA_stats["mean_ranks"].append(np.mean(DYNA_ranks))
    DYNA_stats["mean_cits"].append(np.mean(DYNA_cits))
    DYNA_stats["mean_waste"].append((sim-len(DYNA_ranks))/sim)

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
for stat in DYNA_stats:
    print(stat, np.mean(DYNA_stats[stat]), (np.std(DYNA_stats[stat])))

In [ ]:
DYNA_stats["mean_ites"]

In [ ]:
print("DYNA mean ITE: ", np.mean(DYNA_ites) ,", std:", np.std(DYNA_ites))
print("DYNA mean rank: ", np.mean(DYNA_ranks) ,", std:", np.std(DYNA_ranks))
print("DYNA mean cits alloc: ", np.mean(DYNA_cits_alloc) ,", std:", np.std(DYNA_cits_alloc))
print("DYNA mean cits: ", np.mean(DYNA_cits) ,", std:", np.std(DYNA_cits))
print("DYNA mean waste: ", (sim-len(DYNA_ranks))/sim)

In [ ]:
print("DYNA mean ITE: ", np.mean(DYNA_ites) ,", std:", np.std(DYNA_ites))
print("DYNA mean rank: ", np.mean(DYNA_ranks) ,", std:", np.std(DYNA_ranks))
print("DYNA mean cits alloc: ", np.mean(DYNA_cits_alloc) ,", std:", np.std(DYNA_cits_alloc))
print("DYNA mean cits: ", np.mean(DYNA_cits) ,", std:", np.std(DYNA_cits))
print("DYNA mean waste: ", (sim-len(DYNA_ranks))/sim)

### DynamITE TEST

In [ ]:
def kappa(Q, organ, accuracy = 1):
    organ_dc = organ.copy(deep=True)
    
    ites = []
    p_accs = []
    for rank, patient in Q.iterrows():
        conc = np.concatenate([patient, organ_dc], axis=0)
        conc = np.expand_dims(conc, axis=0)
        accmdl = acceptance_model(conc)
        p_acc , dec_time = accmdl[:,0].numpy().item(), accmdl[:,1].numpy().item()
        organ_dc["COLD_ISCH"] += dec_time
        patient_np = np.array([patient])
        ite = inference_oite.infer(patient_np, np.array([organ_dc])) - inference_oite.infer(patient_np)
        p_accs.append(p_acc)
        ites.append(ite.item())
    
    kappa = 0
    i = 0
    while i in range(int(len(ites)*accuracy)):
        kappa_i = ites[i]*p_accs[i]
        j = 0
        while j in range(i):
            kappa_i *= (1-p_accs[j])
            j += 1
        kappa += kappa_i
        i += 1
    return kappa
        
        
        

In [ ]:
def DynamITE_rank(Q, organ, alpha1 = 0.5, alpha2 = 1, alpha3 = 0.6, iterations = 3, topk = 20, accuracy = 0.4):
    Q = Q.drop(["MELD"], axis = 1)
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    Q_df = pd.DataFrame(Q_np)
    organ_df = pd.DataFrame(organ_np)
    conc = np.concatenate([Q_df, organ_df], axis=1)
    
    #print(conc.shape)
    accmdl = acceptance_model(conc)
    Q["ACC"] , Q["DEC_TIME"] = accmdl[:,0], accmdl[:,1]
    # Calculate ITE
    Q["ITE"] = (inference_oite.infer(Q_np, organ_np) - inference_oite.infer(Q_np)).squeeze()

    Q["DYNA"] = (Q["ACC"]**alpha1) * (Q["ITE"]**alpha2) / (Q["DEC_TIME"]**alpha3)
    #Q["DYNA"] = Q["ITE"]
    # print(Q)
    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='DYNA', ascending=False)

    Q = Q.drop("ITE", axis=1)
    Q = Q.drop("DEC_TIME", axis=1)
    Q = Q.drop("ACC", axis=1)
    Q = Q.drop("DYNA", axis=1)
    #calculate kappa
    current_kappa = kappa(Q, organ, accuracy)

    ##adjacent swaps to evaluate with kappa, if new kappa > current kappa then apply swap: 5 iterations
    for iteration in range(iterations):
        print(f"Iteration {iteration + 1}:")
        # Iterate over adjacent pairs
        for i in range(topk):
            # Swap rows i and i+1
            Q_swapped = Q.copy()
            Q_swapped.iloc[i], Q_swapped.iloc[i + 1] = Q_swapped.iloc[i + 1].copy(), Q_swapped.iloc[i].copy()

            # Calculate kappa for the swapped version
            new_kappa = kappa(Q_swapped, organ, accuracy)

            # If the new kappa is higher, keep the swap
            if new_kappa > current_kappa:
                #print(f"Swapped rows {i} and {i+1} - new kappa: {new_kappa}")
                Q = Q_swapped
                current_kappa = new_kappa
    
    return Q

In [ ]:
sim = 0
DITE_ites = []
DITE_ranks = []
DITE_cits_alloc = []
DITE_cits = []
DITE_waste = 0
for organ in generated_initial_organs.iterrows():
    print(sim)
    organ = organ[1]
    generated_Q = generated_Qs[sim]
    generated_Q = DynamITE_rank(generated_Q, organ, iterations = 1, topk = 5, accuracy = 1)
    rank = 0
    for patient in generated_Q.iterrows():
        rank += 1
        patient = patient[1]
        #combined_input = pd.concat([patient, organ], axis=0)
        #model_output  = acceptance_model(combined_input.to_frame().T)
        
        #accept = model_output[:,0].numpy()
        #print(accept)
        #time = model_output[:,1].numpy()

        accept = synth_acceptance(patient,organ)
        time = synth_time(patient,organ)
        #print("accept", accept)
        #print("time", time)
        
        ## update CIT
        organ["COLD_ISCH"] += time
        #XO = dict(organ) | dict(patient)
        
        if accept > np.random.rand():
            ## organ got accepted: add to obs, break sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
            patient_np = np.array([patient])  # Convert patient Series to NumPy array
            organ_np = np.array([organ])       # Convert organ Series to NumPy array
            #print(patient)
            #print(organ)
            DITE_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
            DITE_ranks.append(rank)
            DITE_cits_alloc.append(organ["COLD_ISCH"])
            break
        #else:
            ## organ got rejected: add to obs, continue sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
    DITE_cits.append(organ["COLD_ISCH"])

    sim +=1

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
dynamite_meta_stats = {"mean ITE": [], "mean rank": [], "mean cits": [], "mean waste": []}

In [ ]:
sim = 0
DITE_ites = []
DITE_ranks = []
DITE_cits_alloc = []
DITE_cits = []
DITE_waste = 0
for organ in generated_initial_organs.iterrows():
    print(sim)
    organ = organ[1]
    generated_Q = generated_Qs[sim]
    generated_Q = DynamITE_rank(generated_Q, organ, iterations = 1, topk = 5, accuracy = 0.4)
    rank = 0
    for patient in generated_Q.iterrows():
        rank += 1
        patient = patient[1]
        #combined_input = pd.concat([patient, organ], axis=0)
        #model_output  = acceptance_model(combined_input.to_frame().T)
        
        #accept = model_output[:,0].numpy()
        #print(accept)
        #time = model_output[:,1].numpy()

        accept = synth_acceptance(patient,organ)
        time = synth_time(patient,organ)
        #print("accept", accept)
        #print("time", time)
        
        ## update CIT
        organ["COLD_ISCH"] += time
        #XO = dict(organ) | dict(patient)
        
        if accept > np.random.rand():
            ## organ got accepted: add to obs, break sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
            patient_np = np.array([patient])  # Convert patient Series to NumPy array
            organ_np = np.array([organ])       # Convert organ Series to NumPy array
            #print(patient)
            #print(organ)
            DITE_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
            DITE_ranks.append(rank)
            DITE_cits_alloc.append(organ["COLD_ISCH"])
            break
        #else:
            ## organ got rejected: add to obs, continue sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
    DITE_cits.append(organ["COLD_ISCH"])

    sim +=1

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
dynamite_meta_stats["mean ITE"].append(np.mean(DITE_ites))
dynamite_meta_stats["mean rank"].append(np.mean(DITE_ranks))
dynamite_meta_stats["mean cits"].append(np.mean(DITE_cits))
dynamite_meta_stats["mean waste"].append((sim-len(DITE_ranks))/sim)

In [ ]:
print(dynamite_meta_stats)

In [ ]:
temp = dynamite_meta_stats

In [ ]:
for stat in dynamite_meta_stats:
    print(stat, np.mean(dynamite_meta_stats[stat]), (np.std(dynamite_meta_stats[stat])))

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
print("DynamITE mean ITE: ", np.mean(DITE_ites) ,", std:", np.std(DITE_ites))
print("DynamITE mean rank: ", np.mean(DITE_ranks) ,", std:", np.std(DITE_ranks))
print("DynamITE mean cits alloc: ", np.mean(DITE_cits_alloc) ,", std:", np.std(DITE_cits_alloc))
print("DynamITE mean cits: ", np.mean(DITE_cits) ,", std:", np.std(DITE_cits))
print("DynamITE mean waste: ", (sim-len(DITE_ranks))/sim)

In [ ]:
sim = 0
ttt_ites = []
ttt_ranks = []
ttt_cits_alloc = []
ttt_cits = []
ttt_waste = 0
for organ in generated_initial_organs.iterrows():
    print(sim)
    organ = organ[1]
    generated_Q = generated_Qs[sim]
    t1 = TIME.time()
    generated_Q = DynamITE_rank(generated_Q, organ, iterations = 3, topk = 20, accuracy = 0.4)
    print(TIME.time() - t1)
    rank = 0
    for patient in generated_Q.iterrows():
        rank += 1
        patient = patient[1]
        #combined_input = pd.concat([patient, organ], axis=0)
        #model_output  = acceptance_model(combined_input.to_frame().T)
        
        #accept = model_output[:,0].numpy()
        #print(accept)
        #time = model_output[:,1].numpy()

        accept = synth_acceptance(patient,organ)
        time = synth_time(patient,organ)
        #print("accept", accept)
        #print("time", time)
        
        ## update CIT
        organ["COLD_ISCH"] += time
        #XO = dict(organ) | dict(patient)
        
        if accept > np.random.rand():
            ## organ got accepted: add to obs, break sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
            patient_np = np.array([patient])  # Convert patient Series to NumPy array
            organ_np = np.array([organ])       # Convert organ Series to NumPy array
            #print(patient)
            #print(organ)
            ttt_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
            ttt_ranks.append(rank)
            ttt_cits_alloc.append(organ["COLD_ISCH"])
            break
        #else:
            ## organ got rejected: add to obs, continue sim
            #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
    ttt_cits.append(organ["COLD_ISCH"])

    sim +=1

### Uncertainty effect

In [ ]:
import matplotlib.pyplot as plt
import time as TIME

### MELD

In [ ]:
distr_acc = []

In [ ]:
meld_meta_stats = {}
meld_meta_stats[1] = meld_stats
for u in [0.8,0.9,1.1,1.2]:
    print("uncertainty lvl:", u )
    #distr_acc_u = []
    meld_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        meld_ites = []
        meld_ranks = []
        meld_cits_alloc = []
        meld_cits = []
        meld_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = MELD_rank(generated_Q)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                #distr_acc_u.append(accept)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    meld_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    #meld_ites.append(ite(patient_np,organ_np))
                    meld_ranks.append(rank)
                    meld_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            meld_cits.append(organ["COLD_ISCH"])
            sim +=1
        meld_u_stats["mean_ites"].append(np.mean(meld_ites))
        meld_u_stats["mean_ranks"].append(np.mean(meld_ranks))
        meld_u_stats["mean_cits"].append(np.mean(meld_cits))
        meld_u_stats["mean_waste"].append((sim-len(meld_ranks))/sim)
    meld_meta_stats[u] = meld_u_stats
    #distr_acc.append(distr_acc_u)

In [ ]:
print(meld_meta_stats)

In [ ]:
# Extract means and standard errors (for confidence intervals)
uncertainty_levels = []
mean_ites_means = []
mean_ites_std_err = []

for key, value in sorted(meld_meta_stats.items()):
    uncertainty_levels.append(key)
    mean_ites = value['mean_ites']
    mean_ites_means.append(np.mean(mean_ites))
    mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

# Plotting
plt.figure(figsize=(10, 6))
plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label='Mean ITES')
plt.xlabel('Uncertainty Level')
plt.ylabel('MELD')
plt.title('MELD Mean ITES Across Uncertainty Levels')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
meldna_meta_stats = {}
meldna_meta_stats[1] = meldna_stats
for u in [0.8,0.9,1.1,1.2]:
    print("uncertainty lvl:", u )
    meldna_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        meldna_ites = []
        meldna_ranks = []
        meldna_cits_alloc = []
        meldna_cits = []
        meldna_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = MELDna_rank(generated_Q)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    meldna_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    
                    meldna_ranks.append(rank)
                    meldna_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            meldna_cits.append(organ["COLD_ISCH"])
            sim +=1
        meldna_u_stats["mean_ites"].append(np.mean(meldna_ites))
        meldna_u_stats["mean_ranks"].append(np.mean(meldna_ranks))
        meldna_u_stats["mean_cits"].append(np.mean(meldna_cits))
        meldna_u_stats["mean_waste"].append((sim-len(meldna_ranks))/sim)
    meldna_meta_stats[u] = meldna_u_stats

In [ ]:
print(meldna_meta_stats)

### MaxAcc

In [ ]:
maxacc_meta_stats = {}
maxacc_meta_stats[1] = maxacc_stats
for u in [0.8,0.9,1.1,1.2]:
    print("uncertainty lvl:", u )
    maxacc_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        maxacc_ites = []
        maxacc_ranks = []
        maxacc_cits_alloc = []
        maxacc_cits = []
        maxacc_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = MaxAcc_rank(generated_Q, organ)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    maxacc_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    
                    maxacc_ranks.append(rank)
                    maxacc_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            maxacc_cits.append(organ["COLD_ISCH"])
            sim +=1
        maxacc_u_stats["mean_ites"].append(np.mean(maxacc_ites))
        maxacc_u_stats["mean_ranks"].append(np.mean(maxacc_ranks))
        maxacc_u_stats["mean_cits"].append(np.mean(maxacc_cits))
        maxacc_u_stats["mean_waste"].append((sim-len(maxacc_ranks))/sim)
    maxacc_meta_stats[u] = maxacc_u_stats

In [ ]:
print(maxacc_meta_stats)

### MinTime

In [ ]:
mintime_meta_stats = {}
mintime_meta_stats[1] = mintime_stats
for u in [0.8,0.9,1.1,1.2]:
    print("uncertainty lvl:", u )
    mintime_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        mintime_ites = []
        mintime_ranks = []
        mintime_cits_alloc = []
        mintime_cits = []
        mintime_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = MinTime_rank(generated_Q, organ)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    mintime_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    
                    mintime_ranks.append(rank)
                    mintime_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            mintime_cits.append(organ["COLD_ISCH"])
            sim +=1
        mintime_u_stats["mean_ites"].append(np.mean(mintime_ites))
        mintime_u_stats["mean_ranks"].append(np.mean(mintime_ranks))
        mintime_u_stats["mean_cits"].append(np.mean(mintime_cits))
        mintime_u_stats["mean_waste"].append((sim-len(mintime_ranks))/sim)
    mintime_meta_stats[u] = mintime_u_stats

In [ ]:
print(mintime_meta_stats)

### TB

In [ ]:
TB_meta_stats = {1.25: TB_meta_stats[1.25], 1.5: TB_meta_stats[1.5]}

In [ ]:
TB_meta_stats = {}
TB_meta_stats[1] = TB_stats
for u in [0.8,0.9,1.1,1.2]:
    print("uncertainty lvl:", u )
    TB_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        TB_ites = []
        TB_ranks = []
        TB_cits_alloc = []
        TB_cits = []
        TB_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = TB_rank(generated_Q, organ)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    TB_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    
                    TB_ranks.append(rank)
                    TB_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            TB_cits.append(organ["COLD_ISCH"])
            sim +=1
        TB_u_stats["mean_ites"].append(np.mean(TB_ites))
        TB_u_stats["mean_ranks"].append(np.mean(TB_ranks))
        TB_u_stats["mean_cits"].append(np.mean(TB_cits))
        TB_u_stats["mean_waste"].append((sim-len(TB_ranks))/sim)
    TB_meta_stats[u] = TB_u_stats

In [ ]:
print(TB_meta_stats)

In [ ]:
# Extract means and standard errors (for confidence intervals)
uncertainty_levels = []
mean_ites_means = []
mean_ites_std_err = []

for key, value in sorted(TB_meta_stats.items()):
    uncertainty_levels.append(key)
    mean_ites = value['mean_ites']
    mean_ites_means.append(np.mean(mean_ites))
    mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

# Plotting
plt.figure(figsize=(10, 6))
plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label='Mean ITES')
plt.xlabel('Uncertainty Level')
plt.ylabel('MELD')
plt.title('MELD Mean ITES Across Uncertainty Levels')
plt.grid(True)
plt.legend()
plt.show()

### Dyna

In [ ]:
def Dyna_rank(Q, organ, alpha1 = 0.5, alpha2 = 1, alpha3 = 0.6, iterations=5, topk = 10):
    Q = Q.drop(["MELD"], axis = 1)
    # Ensure Q and organ are in the correct format for inference_oite.infer()
    Q_np = Q.to_numpy()  # Convert Q to a NumPy array
    n_patients = Q.shape[0]
    organ_np = np.tile(organ, (n_patients, 1))   # Convert organ to NumPy array
    Q_df = pd.DataFrame(Q_np)
    organ_df = pd.DataFrame(organ_np)
    conc = np.concatenate([Q_df, organ_df], axis=1)
    
    #print(conc.shape)
    accmdl = acceptance_model(conc)
    Q["ACC"] , Q["DEC_TIME"] = accmdl[:,0], accmdl[:,1]
    # Calculate ITE
    Q["ITE"] = (inference_oite.infer(Q_np, organ_np) - inference_oite.infer(Q_np)).squeeze()

    #Q["DYNA"] = Q["ACC"] * Q["ITE"] 
    Q["DYNA"] = (Q["ACC"]**alpha1) * (Q["ITE"]**alpha2) / (Q["DEC_TIME"]**alpha3)
    
    # print(Q)
    # Sort by ITE and drop the column after sorting
    Q = Q.sort_values(by='DYNA', ascending=False)

    Q = Q.drop("ITE", axis=1)
    Q = Q.drop("DEC_TIME", axis=1)
    Q = Q.drop("ACC", axis=1)
    Q = Q.drop("DYNA", axis=1)
    
    return Q

In [ ]:
#DYNA_meta_stats = {}
DYNA_meta_stats[1] = DYNA_stats
for u in [1.2]:
    print("uncertainty lvl:", u )
    DYNA_u_stats = {"mean_ites": [], "mean_ranks": [] , "mean_cits": [] , "mean_waste": []}
    for run in range(10):
        print("run:",run)
        sim = 0
        DYNA_ites = []
        DYNA_ranks = []
        DYNA_cits_alloc = []
        DYNA_cits = []
        DYNA_waste = 0
        for organ in generated_initial_organs.iterrows():
            #print(sim)
            organ = organ[1]
            generated_Q = generated_Qs[sim]
            generated_Q = Dyna_rank(generated_Q, organ)
            rank = 0
            for patient in generated_Q.iterrows():
                rank += 1
                patient = patient[1]
                #combined_input = pd.concat([patient, organ], axis=0)
                #model_output  = acceptance_model(combined_input.to_frame().T)
                
                #accept = model_output[:,0].numpy()
                #print(accept)
                #time = model_output[:,1].numpy()
        
                accept = synth_acceptance(patient,organ, uncertainty = u)
                time = synth_time(patient,organ)
                #print("accept", accept)
                #print("time", time)
                
                ## update CIT
                organ["COLD_ISCH"] += time
                #XO = dict(organ) | dict(patient)
                
                if accept > np.random.rand():
                    ## organ got accepted: add to obs, break sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 1, "DEC_TIME": time}
                    patient_np = np.array([patient])  # Convert patient Series to NumPy array
                    organ_np = np.array([organ])       # Convert organ Series to NumPy array
                    #print(patient)
                    #print(organ)
                    DYNA_ites.append((inference_oite.infer(patient_np, organ_np) - inference_oite.infer(patient_np)).item())
                    
                    DYNA_ranks.append(rank)
                    DYNA_cits_alloc.append(organ["COLD_ISCH"])
                    break
                #else:
                    ## organ got rejected: add to obs, continue sim
                    #simulated_acceptance_dataset.loc[len(simulated_acceptance_dataset)] = XO | {"ACC": 0, "DEC_TIME": time}
            DYNA_cits.append(organ["COLD_ISCH"])
            sim +=1
        DYNA_u_stats["mean_ites"].append(np.mean(DYNA_ites))
        DYNA_u_stats["mean_ranks"].append(np.mean(DYNA_ranks))
        DYNA_u_stats["mean_cits"].append(np.mean(DYNA_cits))
        DYNA_u_stats["mean_waste"].append((sim-len(DYNA_ranks))/sim)
    DYNA_meta_stats[u] = DYNA_u_stats

In [ ]:
DYNA_meta_stats[1] = DYNA_stats

In [ ]:
sorted(DYNA_meta_stats.items())

In [ ]:
print(DYNA_meta_stats)

In [ ]:
for stat in DYNA_meta_stats[1]:
    print(stat, np.mean(DYNA_meta_stats[1][stat]), (np.std(DYNA_meta_stats[1][stat])))

In [ ]:
# Extract means and standard errors (for confidence intervals)
uncertainty_levels = []
mean_ites_means = []
mean_ites_std_err = []

for key, value in sorted(DYNA_meta_stats.items()):
    uncertainty_levels.append(key)
    mean_ites = value['mean_ites']
    mean_ites_means.append(np.mean(mean_ites))
    mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

# Plotting
plt.figure(figsize=(10, 6))
plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label='Mean ITES')
plt.xlabel('Uncertainty Level')
plt.ylabel('MELD')
plt.title('MELD Mean ITES Across Uncertainty Levels')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
exp3data = {"MELD": meld_meta_stats, "MELD-Na": meldna_meta_stats, "MaxAcc": maxacc_meta_stats, "MinTime":mintime_meta_stats, "TB":TB_meta_stats, "DynamITE": DYNA_meta_stats}

In [ ]:
exp3data1 = {"MELD": meld_meta_stats, "MELD-Na": meldna_meta_stats, "MaxAcc": maxacc_meta_stats, "MinTime":mintime_meta_stats}
exp3data2 = {"TB":TB_meta_stats, "DynamITE": DYNA_meta_stats}

### ITES

In [ ]:
# Create a figure with two Y-axes using the `subplot` method for the broken axis
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [1, 2]})

# Define Y-limits for the two parts
y_limits_top = (3200, 4200)
y_limits_bottom = (500, 1200)

# Plotting data
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ites_means = []
    mean_ites_std_err = []

    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ites = value['mean_ites']
        mean_ites_means.append(np.mean(mean_ites))
        mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))

    # Plot on both axes
    ax1.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)
    ax2.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)

# Set limits for both axes
ax1.set_ylim(y_limits_top)
ax2.set_ylim(y_limits_bottom)

# Hide the spines between the two subplots
ax1.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
#ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # Hide x-axis labels on the top plot
#ax2.xaxis.tick_bottom()

# Add a diagonal line to represent the broken axis
d = .015  # Size of diagonal lines
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # Top-left diagonal
ax1.plot((1- d, 1+ d), (-d, +d), **kwargs)  # Top-right diagonal

kwargs.update(transform=ax2.transAxes)  # Switch to bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # Bottom-left diagonal
ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # Bottom-right diagonal

# Add labels and title
ax2.set_xlabel('Uncertainty Level')
ax2.set_ylabel('Mean ITEs')
fig.suptitle('Mean ITEs Across Uncertainty Levels for Multiple Policies')

# Add grid and legend
ax2.grid(True)
ax1.grid(True)
ax2.legend()

plt.show()

In [ ]:
# Assume exp3data is already defined with your data

# Create a figure with two Y-axes using the `subplot` method for the broken axis
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [1, 2]})

# Define Y-limits for the two parts
y_limits_top = (3400, 3650)
y_limits_bottom = (550, 1150)

# Plotting data
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ites_means = []
    mean_ites_std_err = []

    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ites = value['mean_ites']
        mean_ites_means.append(np.mean(mean_ites))
        mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))

    # Plot on both axes
    ax1.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)
    ax2.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)

# Set limits for both axes
ax1.set_ylim(y_limits_top)
ax2.set_ylim(y_limits_bottom)

# Hide the spines between the two subplots
ax1.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax1.tick_params(labeltop=False)  # Hide x-axis labels on the top plot

# Remove unwanted ticks on the top plot
ax2.xaxis.set_ticks([0.8, 0.9, 1, 1.1, 1.2])

# Add a diagonal line to represent the broken axis
d = .015  # Size of diagonal lines
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # Top-left diagonal
ax1.plot((1- d, 1+ d), (-d, +d), **kwargs)  # Top-right diagonal

kwargs.update(transform=ax1.transAxes, color='k', clip_on=False)  # Switch to bottom axes
ax2.plot((-d, +d), (-0.26 - d, -0.26 + d), **kwargs)  # Bottom-left diagonal
ax2.plot((1 - d, 1 + d), (-0.26 - d, -0.26 + d), **kwargs)  # Bottom-right diagonal

# Adjust layout: reduce white space between title and plot
fig.subplots_adjust(top=0.88)

# Add labels and title
ax2.set_xlabel('Uncertainty Level')
ax2.set_ylabel('Mean ITE')
fig.suptitle('Mean ITE Across Uncertainty Levels for Multiple Policies', x= 0.4, y=0.95)

# Add grid and legend
ax2.grid(True)
ax1.grid(True)
# Move the legend outside the figure
ax2.legend(loc='upper left', bbox_to_anchor=(1, 1), title="Policies")

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Leave space on the right for the legend
plt.savefig('uncertainty ITEs.svg', format='svg', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Initialize subplots
fig, axs = plt.subplots(3, 1, figsize=(5, 6))

# Plot mean_ranks
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ranks_means = []
    mean_ranks_std_err = []

    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ranks = value['mean_ranks']
        mean_ranks_means.append(np.mean(mean_ranks))
        mean_ranks_std_err.append(np.std(mean_ranks) / np.sqrt(len(mean_ranks)))

    axs[0].errorbar(uncertainty_levels, mean_ranks_means, yerr=mean_ranks_std_err, fmt='-o', capsize=5, label=policy_name)

axs[0].set_title('Mean APTR Across Uncertainty Levels for Multiple Policies')
axs[0].set_ylabel('Mean APTR')
axs[0].grid(True)
axs[0].legend(loc='upper left', bbox_to_anchor=(1, 1), title="Policies")

# Plot mean_cits
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_cits_means = []
    mean_cits_std_err = []

    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_cits = value['mean_cits']
        mean_cits_means.append(np.mean(mean_cits))
        mean_cits_std_err.append(np.std(mean_cits) / np.sqrt(len(mean_cits)))

    axs[1].errorbar(uncertainty_levels, mean_cits_means, yerr=mean_cits_std_err, fmt='-o', capsize=5, label=policy_name)

axs[1].set_title('Mean CIT Across Uncertainty Levels for Multiple Policies')
axs[1].set_ylabel('Mean CIT')
axs[1].grid(True)

# Plot mean_waste
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_waste_means = []
    mean_waste_std_err = []

    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_waste = value['mean_waste']
        mean_waste_means.append(np.mean(mean_waste))
        mean_waste_std_err.append(np.std(mean_waste) / np.sqrt(len(mean_waste)))

    axs[2].errorbar(uncertainty_levels, mean_waste_means, yerr=mean_waste_std_err, fmt='-o', capsize=5, label=policy_name)

axs[2].set_title('Mean Nonuse Across Uncertainty Levels for Multiple Policies')
axs[2].set_ylabel('Mean Nonuse')
axs[2].set_xlabel('Uncertainty Level')
axs[2].grid(True)

# Final adjustments
plt.tight_layout()
plt.savefig('uncertainty_metrics.svg', format='svg', bbox_inches='tight', dpi=300)
plt.show()

### RANKS

In [ ]:
plt.figure(figsize=(10, 6))

# Loop through each policy
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ites_means = []
    mean_ites_std_err = []

    # Extract means and standard errors for each uncertainty level
    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ites = value['mean_ranks']
        mean_ites_means.append(np.mean(mean_ites))
        mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

    # Plot for the current policy
    plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)

# Adding labels and title
# Remove unwanted ticks on the top plot
plt.xlabel('Uncertainty Level')
plt.ylabel('Mean APTR')
plt.title('Mean APTR Across Uncertainty Levels for Multiple Policies')
# Set the x-axis ticks manually
plt.xticks([0.8, 0.9, 1, 1.1, 1.2])
plt.grid(True)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('uncertainty APTR2.svg', format='svg', bbox_inches='tight', dpi=300)
plt.show()

### CITS

In [ ]:
plt.figure(figsize=(10, 6))

# Loop through each policy
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ites_means = []
    mean_ites_std_err = []

    # Extract means and standard errors for each uncertainty level
    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ites = value['mean_cits']
        mean_ites_means.append(np.mean(mean_ites))
        mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

    # Plot for the current policy
    plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)

# Adding labels and title
# Remove unwanted ticks on the top plot
plt.xlabel('Uncertainty Level')
plt.ylabel('Mean CIT')
plt.title('Mean CIT Across Uncertainty Levels for Multiple Policies')
# Set the x-axis ticks manually
plt.xticks([0.8, 0.9, 1, 1.1, 1.2])
plt.grid(True)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('uncertainty CITs2.svg', format='svg', bbox_inches='tight', dpi=300)
plt.show()

### WASTE

In [ ]:
plt.figure(figsize=(10, 6))

# Loop through each policy
for policy_name, data in exp3data.items():
    uncertainty_levels = []
    mean_ites_means = []
    mean_ites_std_err = []

    # Extract means and standard errors for each uncertainty level
    for key, value in sorted(data.items()):
        uncertainty_levels.append(key)
        mean_ites = value['mean_waste']
        mean_ites_means.append(np.mean(mean_ites))
        mean_ites_std_err.append(np.std(mean_ites) / np.sqrt(len(mean_ites)))  # Standard error

    # Plot for the current policy
    plt.errorbar(uncertainty_levels, mean_ites_means, yerr=mean_ites_std_err, fmt='-o', capsize=5, label=policy_name)

# Adding labels and title
# Remove unwanted ticks on the top plot
plt.xlabel('Uncertainty Level')
plt.ylabel('Mean Nonuse percentage')
plt.title('Mean Nonuse percentage Across Uncertainty Levels for Multiple Policies')
# Set the x-axis ticks manually
plt.xticks([0.8, 0.9, 1, 1.1, 1.2])
plt.grid(True)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('uncertainty nonuse2.svg', format='svg', bbox_inches='tight', dpi=300)
plt.show()